# Field Shift — rotação de culturas com dados da NASA (protótipo)

**Desafio NASA Space Apps 2026 — *Field Shift: Adapting Farms with NASA Data*.**
Ferramenta de apoio à decisão que junta **dados de observação da Terra da NASA (NASA POWER)**, o **solo do local**,
as **características das culturas** e as **prioridades do agricultor** para sugerir rotações que protejam o solo
e se adaptem ao clima que está mudando.

Como funciona:
1. Baixa a série **diária** do **NASA POWER** (chuva, temperatura máx./mín.) para um ponto (padrão: Uberlândia-MG).
2. Calcula a evapotranspiração (Hargreaves) e roda um **balanço hídrico tipo "balde"**, inspirado no modelo SARRA usado pelo ZARC.
3. Para **6 culturas** (soja, milho safrinha, sorgo, feijão, crotalária e braquiária), **36 decêndios de semeadura** e
   **6 níveis de retenção de água do solo** (classes AD1–AD6 do ZARC; o produtor pode informar só o tipo: arenoso, médio, argiloso ou muito argiloso), calcula o **ISNA** por fase e o **risco climático**
   (frequência de anos em que a cultura sofreu falta de água numa fase crítica).
4. **Field Shift:** compara o risco entre **1991–2008** e **2009–2024** para mostrar quais janelas ficaram mais arriscadas.
5. **Recomendador de rotação:** compara pelo menos três estratégias, **incluindo a rotação atual** do agricultor, em **cartões
   verde/amarelo/vermelho** com risco climático, uso de água, calor, geada, benefícios, pontos de atenção e lacunas.

👉 **Para usar:** edite só a célula **"Dados do produtor"** logo abaixo e rode tudo (*Run All*).

⚠️ Durações das fases, Kc, limites de ISNA e notas de saúde do solo são **provisórios**: calibrar com a documentação do ZARC
e com agrônomos antes de qualquer uso real.

## Como ler os resultados

**Esta ferramenta apoia, não decide.** Ela mostra padrões do clima dos últimos 34 anos e compara opções de rotação.
A decisão final é do produtor, de preferência junto com um agrônomo que conheça a região.

| Termo | O que quer dizer |
|---|---|
| **Evapotranspiração (ETo, ETm)** | Água que sai do solo e da planta para o ar. A ETo é a de uma grama de referência; a ETm é a que a cultura pediria se não faltasse água. |
| **Balanço hídrico** | Conta diária de "entra chuva, sai evapotranspiração", como um balde de água no solo. |
| **ISNA** | Quanto da água que a planta precisava ela de fato conseguiu (0 a 1). Abaixo do limite numa fase crítica, contamos falha. |
| **Risco** | Em quantos anos, de cada 100, a lavoura teria sofrido falta de água plantando naquela época. |
| **Classes AD1–AD6** | Quanta água o solo segura, da mais baixa (AD1) à mais alta (AD6), como no ZARC. |
| **Decêndio** | Período de 10 dias. O ano tem 36. |
| **Field shift** | Quanto o risco mudou entre 1991–2008 e 2009–2024. |

**Cores dos cartões:** 🟢 risco até 20% (pode plantar) · 🟡 de 20% a 40% (plantar com cuidado) · 🔴 acima de 40% (evite),
seguindo as faixas do ZARC. A **ordem** dos cartões usa a nota final, que também pesa a saúde do solo e a diversificação.

**Dados de satélite × medição na fazenda:** o NASA POWER estima o clima numa grade de cerca de 50 km, combinando satélites
e modelos. Ele cobre o mundo todo e tem séries longas, mas pode errar a chuva de uma lavoura específica.
Um pluviômetro na fazenda é mais preciso no local; usar os dois juntos dá o melhor resultado.

In [ ]:
# ===== Dados do produtor (edite aqui) =====
MUNICIPIO = "Uberlândia-MG"
LAT, LON = -18.9186, -48.2772

HISTORICO = ["soja", "milho_safrinha"]   # o que foi plantado, do mais antigo ao mais recente
SOLO = "argiloso"                        # tipo de solo: "arenoso", "médio", "argiloso", "muito argiloso"
                                         # ou, se souber pela análise de solo, a classe do ZARC: "AD1" ... "AD6"
DATA_PLANTIO = "2026-10-21"              # quando pretende plantar a próxima safra
PRIORIDADES = {                          # quanto cada objetivo pesa (não precisa somar 1)
    "risco": 0.5,         # menos chance de perder a lavoura por falta de chuva
    "solo": 0.3,          # mais saúde do solo (palha, nitrogênio, raízes)
    "diversidade": 0.2,   # variar as culturas em relação ao histórico
}

# ===== Período dos dados =====
ANO_INI, ANO_FIM = 1991, 2025
PERIODO_1 = range(1991, 2009)   # 1991–2008
PERIODO_2 = range(2009, 2025)   # 2009–2024 (safras com ciclo completo nos dados)

In [ ]:
# ===== Download NASA POWER (diário, comunidade AG) =====
import requests, pandas as pd, numpy as np

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "PRECTOTCORR,T2M_MAX,T2M_MIN",
    "community": "AG",
    "latitude": LAT, "longitude": LON,
    "start": f"{ANO_INI}0101", "end": f"{ANO_FIM}1231",
    "format": "JSON",
}
r = requests.get(url, params=params, timeout=120)
r.raise_for_status()
p = r.json()["properties"]["parameter"]

df = pd.DataFrame({
    "chuva": pd.Series(p["PRECTOTCORR"]),
    "tmax": pd.Series(p["T2M_MAX"]),
    "tmin": pd.Series(p["T2M_MIN"]),
})
df.index = pd.to_datetime(df.index, format="%Y%m%d")
df = df.replace(-999, np.nan)
print("Dias faltantes:\n", df.isna().sum())
df = df.interpolate(limit=5).fillna({"chuva": 0})
df.describe()

In [ ]:
# ===== Funções do modelo (balanço hídrico, ISNA e risco) =====
import numpy as np
import pandas as pd

# ---------------- Solo: água disponível por classe ----------------
# ZARC Soja MG 2025/2026 (Portaria MAPA): raiz efetiva de 60 cm e CAD de 24, 32, 42, 55, 72 e 95 mm (AD1–AD6).
# Guardamos em mm por cm de solo (CAD / 60) para aplicar a outras profundidades de raiz.
CAD_ZARC_SOJA = {"AD1": 24, "AD2": 32, "AD3": 42, "AD4": 55, "AD5": 72, "AD6": 95}
CLASSES_AD = {k: v / 60 for k, v in CAD_ZARC_SOJA.items()}
# O ZARC não dá nome de textura às classes; os nomes abaixo descrevem a retenção de água.
NOMES_AD = {
    "AD1": "AD1 – retenção muito baixa",
    "AD2": "AD2 – retenção baixa",
    "AD3": "AD3 – retenção média",
    "AD4": "AD4 – retenção média-alta",
    "AD5": "AD5 – retenção alta",
    "AD6": "AD6 – retenção muito alta",
}
# O produtor normalmente sabe o tipo de solo, não a classe AD. Tradução APROXIMADA (o ZARC define a classe pela
# análise granulométrica — areia, silte e argila —, então o ideal é usar a classe AD da análise de solo):
TEXTURA_PARA_AD = {"arenoso": "AD2", "médio": "AD3", "medio": "AD3", "argiloso": "AD4", "muito argiloso": "AD5"}


def resolver_solo(solo):
    s = solo.strip()
    if s.upper() in CLASSES_AD:
        return s.upper()
    ad = TEXTURA_PARA_AD.get(s.lower())
    if ad is None:
        raise ValueError(f"Solo '{solo}' não reconhecido. Use {list(TEXTURA_PARA_AD)} ou AD1–AD6.")
    print(f"Solo '{s}' tratado como {NOMES_AD[ad]} (aproximação; confirme com a análise de solo).")
    return ad


SOLO_AD = resolver_solo(SOLO)
P_DEPLECAO = 0.5   # fração da CAD consumida sem estresse

# Milho 2ª safra, feijão e sorgo ainda usam no ZARC a CAD por TEXTURA (tipo 1 arenoso, 2 médio, 3 argiloso).
# Valores das portarias: milho 2ª safra DF (raiz 45 cm), feijão 1ª safra MG, sorgo granífero (raiz 50 cm).
CAD_ZARC_TEXTURA = {"milho_safrinha": (30, 47, 72), "feijao": (28, 44, 60), "sorgo": (35, 55, 75)}
TIPO_DA_CLASSE = {"AD1": 0, "AD2": 0, "AD3": 1, "AD4": 2, "AD5": 2, "AD6": 2}  # AD → tipo 1, 2 ou 3


def cad_mm(cultura, classe):
    """Água disponível (mm) que a raiz da cultura alcança naquele solo."""
    if cultura in CAD_ZARC_TEXTURA:
        return CAD_ZARC_TEXTURA[cultura][TIPO_DA_CLASSE[classe]]
    return CLASSES_AD[classe] * CULTURAS[cultura]["raiz_cm"]

# ---------------- Culturas ----------------
# Ciclos, limites de ISNA, raiz e temperaturas: portarias do ZARC (MAPA/Embrapa) — soja MG 2025/26 (ciclos 100/115/130 d),
# milho 2ª safra DF (até 110 d), feijão 1ª safra MG (70/80/90 d), sorgo granífero (até 110 d).
# As portarias não trazem a duração de cada fase nem o Kc: dividimos o ciclo em fases e usamos Kc da FAO-56.
# Crotalária e braquiária não têm ZARC: valores ESTIMADOS (FAO-56 e literatura), marcados como tal.
# saude_solo (0–1): palha, raízes e nitrogênio (julgamento agronômico geral; validar com agrônomo).
CULTURAS = {
    "soja": {"nome": "Soja (115 dias)", "base": "soja", "familia": "leguminosa", "papel": "grão", "saude_solo": 0.55,
             "raiz_cm": 60, "calor": 35, "geada": 3, "fonte": "ZARC",
             "fases": [("F1_inicial", 15, 0.40), ("F2_vegetativo", 35, 0.80),
                       ("F3_reprodutivo", 45, 1.15), ("F4_maturacao", 20, 0.50)],
             "limites": {"F1_inicial": 0.50, "F3_reprodutivo": 0.55}},
    "soja_precoce": {"nome": "Soja precoce (100 dias)", "base": "soja", "familia": "leguminosa", "papel": "grão",
             "saude_solo": 0.55, "raiz_cm": 60, "calor": 35, "geada": 3, "fonte": "ZARC",
             "fases": [("F1_inicial", 15, 0.40), ("F2_vegetativo", 30, 0.80),
                       ("F3_reprodutivo", 40, 1.15), ("F4_maturacao", 15, 0.50)],
             "limites": {"F1_inicial": 0.50, "F3_reprodutivo": 0.55}},
    "milho_safrinha": {"nome": "Milho safrinha", "base": "milho_safrinha", "familia": "gramínea", "papel": "grão",
             "saude_solo": 0.65, "raiz_cm": 45, "calor": 35, "geada": 2, "fonte": "ZARC",
             "fases": [("F1_inicial", 15, 0.30), ("F2_vegetativo", 35, 0.80),
                       ("F3_reprodutivo", 40, 1.20), ("F4_maturacao", 20, 0.60)],
             "limites": {"F1_inicial": 0.60, "F3_reprodutivo": 0.50}},
    "sorgo": {"nome": "Sorgo", "base": "sorgo", "familia": "gramínea", "papel": "grão", "saude_solo": 0.65,
             "raiz_cm": 50, "calor": 38, "geada": 2, "fonte": "ZARC",
             "fases": [("F1_inicial", 15, 0.30), ("F2_vegetativo", 35, 0.75),
                       ("F3_reprodutivo", 40, 1.05), ("F4_maturacao", 20, 0.55)],
             "limites": {"F1_inicial": 0.60, "F3_reprodutivo": 0.45}},
    "feijao": {"nome": "Feijão", "base": "feijao", "familia": "leguminosa", "papel": "grão", "saude_solo": 0.45,
             "raiz_cm": 40, "calor": 32, "geada": 3, "fonte": "ZARC",
             "fases": [("F1_inicial", 10, 0.40), ("F2_vegetativo", 25, 0.75),
                       ("F3_reprodutivo", 30, 1.15), ("F4_maturacao", 15, 0.35)],
             "limites": {"F1_inicial": 0.50, "F3_reprodutivo": 0.60}},
    "crotalaria": {"nome": "Crotalária (cobertura)", "base": "crotalaria", "familia": "leguminosa", "papel": "cobertura",
             "saude_solo": 0.90, "raiz_cm": 60, "calor": 38, "geada": 3, "fonte": "estimativa",
             "fases": [("F1_inicial", 15, 0.40), ("F2_vegetativo", 35, 0.80), ("F3_reprodutivo", 40, 1.00)],
             "limites": {"F1_inicial": 0.45}},
    "braquiaria": {"nome": "Braquiária (cobertura)", "base": "braquiaria", "familia": "gramínea", "papel": "cobertura",
             "saude_solo": 0.90, "raiz_cm": 60, "calor": 40, "geada": 2, "fonte": "estimativa",
             "fases": [("F1_inicial", 20, 0.40), ("F2_vegetativo", 80, 0.85)],
             "limites": {"F1_inicial": 0.40}},
}

def radiacao_extraterrestre(lat_graus, doy):
    """Ra em mm/dia equivalentes (FAO-56)."""
    phi = np.radians(lat_graus)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    ra_mj = (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))
    return 0.408 * ra_mj


def eto_hargreaves(tmax, tmin, lat, doy):
    tmed = (tmax + tmin) / 2
    ra = radiacao_extraterrestre(lat, doy)
    return np.maximum(0.0023 * (tmed + 17.8) * np.sqrt(np.maximum(tmax - tmin, 0)) * ra, 0)


def _passo_balde(S, chuva, etm, cad):
    """Um dia do balanço hídrico. Funciona com números ou vetores numpy."""
    S = np.minimum(cad, S + chuva)
    limiar = (1 - P_DEPLECAO) * cad
    ks = np.where(S >= limiar, 1.0, S / limiar)
    etr = np.minimum(etm * ks, S)
    return S - etr, etr


def isna_por_fase(chuva, eto, i0, cad, fases, spinup=30, kc_pousio=0.3):
    """Simula, de uma vez, várias semeaduras (vetores i0 e cad do mesmo tamanho).
    Retorna ({fase: vetor de ISNA}, vetor com a água que a cultura demanda no ciclo, em mm). Vetorizado: leve em memória e rápido mesmo em computador simples."""
    i0, cad = np.asarray(i0), np.asarray(cad, dtype=float)
    S = 0.5 * cad
    for t in range(-spinup, 0):  # aquecimento com solo em pousio
        S, _ = _passo_balde(S, chuva[i0 + t], kc_pousio * eto[i0 + t], cad)
    out, t = {}, 0
    etm_total = np.zeros_like(cad)
    for nome, dias, kc in fases:
        s_etr = np.zeros_like(cad)
        s_etm = np.zeros_like(cad)
        for _ in range(dias):
            etm = kc * eto[i0 + t]
            S, etr = _passo_balde(S, chuva[i0 + t], etm, cad)
            s_etr += etr
            s_etm += etm
            etm_total += etm
            t += 1
        out[nome] = np.divide(s_etr, s_etm, out=np.ones_like(s_etr), where=s_etm > 0)
    return out, etm_total


def datas_decendios(ano):
    return [pd.Timestamp(ano, m, d) for m in range(1, 13) for d in (1, 11, 21)]


def simular_risco(df, lat, anos, cultura="soja", spinup=30):
    """df: índice diário (DatetimeIndex) com colunas chuva, tmax, tmin.
    Retorna uma linha por ano × decêndio × classe de solo com o ISNA e se houve falha."""
    cfg = CULTURAS[cultura]
    df = df.sort_index()
    eto = eto_hargreaves(df.tmax.values, df.tmin.values, lat, df.index.day_of_year.values)
    chuva = df.chuva.values
    pos = pd.Series(np.arange(len(df)), index=df.index)
    n_ciclo = sum(d for _, d, _ in cfg["fases"])

    comb = pd.DataFrame([(c, a, k, d) for c in CLASSES_AD for a in anos
                         for k, d in enumerate(datas_decendios(a), start=1)],
                        columns=["classe_solo", "ano", "decendio", "data_semeadura"])
    comb["i0"] = comb.data_semeadura.map(pos)
    comb = comb.dropna(subset=["i0"])
    comb["i0"] = comb.i0.astype(int)
    comb = comb[(comb.i0 - spinup >= 0) & (comb.i0 + n_ciclo <= len(df))].reset_index(drop=True)

    cad = comb.classe_solo.map(lambda c: cad_mm(cultura, c)).values.astype(float)
    isna, etm_total = isna_por_fase(chuva, eto, comb.i0.values, cad, cfg["fases"], spinup=spinup)
    comb["agua_mm"] = etm_total
    for fase, v in isna.items():
        comb[f"isna_{fase}"] = v
    comb["falha"] = np.logical_or.reduce([isna[f] < lim for f, lim in cfg["limites"].items()])
    comb["cultura"] = cultura
    return comb.drop(columns="i0")


def classificar(risco):
    if risco <= 0.20:
        return "20%"
    if risco <= 0.30:
        return "30%"
    if risco <= 0.40:
        return "40%"
    return "não indicado"


def tabua_risco(sim, anos=None):
    s = sim if anos is None else sim[sim.ano.isin(anos)]
    t = s.groupby(["cultura", "classe_solo", "decendio"], as_index=False).agg(
        risco=("falha", "mean"), n_anos=("ano", "nunique"))
    t["faixa"] = t.risco.apply(classificar)
    return t

In [ ]:
# ===== Simulação para todas as culturas e todos os anos =====
sim = pd.concat([simular_risco(df, LAT, range(ANO_INI, ANO_FIM), c) for c in CULTURAS], ignore_index=True)
tabua = tabua_risco(sim)
tabua.assign(classe_solo=tabua.classe_solo.map(NOMES_AD)).to_csv("tabua_risco_nasa_power.csv", index=False)
print(sim.shape)

# Tábua da soja no formato do ZARC: decêndio × nível de retenção de água do solo
(tabua[tabua.cultura == "soja"]
 .assign(classe_solo=lambda t: t.classe_solo.map(NOMES_AD))
 .pivot(index="decendio", columns="classe_solo", values="faixa"))

In [ ]:
# ===== Field Shift: o risco mudou entre os dois períodos? =====
import matplotlib.pyplot as plt

t1 = tabua_risco(sim, PERIODO_1).assign(periodo="1991–2008")
t2 = tabua_risco(sim, PERIODO_2).assign(periodo="2009–2024")
comp = t1.merge(t2, on=["cultura", "classe_solo", "decendio"], suffixes=("_p1", "_p2"))
comp["variacao_pp"] = (comp.risco_p2 - comp.risco_p1) * 100
comp.assign(classe_solo=comp.classe_solo.map(NOMES_AD)).to_csv("comparacao_periodos.csv", index=False)

meses = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
rotulos = [f"{m}-{d}" for m in meses for d in ("1","2","3")]

c = comp[(comp.cultura == "soja") & (comp.classe_solo == SOLO_AD)]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(c.decendio, c.risco_p1 * 100, label="1991–2008")
ax.plot(c.decendio, c.risco_p2 * 100, label="2009–2024")
for lim in (20, 30, 40):
    ax.axhline(lim, ls=":", lw=0.8, color="gray")
ax.set_xticks(range(1, 37), rotulos, rotation=90, fontsize=7)
ax.set_ylabel("Risco de falha (%)")
ax.set_title(f"Soja (ciclo médio) — {MUNICIPIO} — {NOMES_AD[SOLO_AD]}: risco por decêndio de semeadura")
ax.legend(); plt.tight_layout(); plt.show()

piv = comp[comp.cultura == "soja"].pivot(index="classe_solo", columns="decendio", values="variacao_pp")
fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(piv.values, cmap="RdBu_r", vmin=-30, vmax=30, aspect="auto")
ax.set_yticks(range(len(piv.index)), [NOMES_AD[i] for i in piv.index])
ax.set_xticks(range(36), rotulos, rotation=90, fontsize=7)
fig.colorbar(im, label="Variação do risco (p.p.)")
ax.set_title("Soja: mudança no risco entre períodos (vermelho = ficou mais arriscado)")
plt.tight_layout(); plt.show()

In [ ]:
# ===== Tendência da chuva na estação (out–mar) por safra =====
from scipy.stats import kendalltau

d = df.copy()
d["safra"] = np.where(d.index.month >= 7, d.index.year, d.index.year - 1)
est = d[d.index.month.isin([10, 11, 12, 1, 2, 3])].groupby("safra").chuva.sum()
est = est.loc[ANO_INI:ANO_FIM - 1]
tau, pval = kendalltau(est.index, est.values)
incl = np.polyfit(est.index, est.values, 1)[0]
print(f"Chuva out–mar: tendência {incl:+.1f} mm/ano | Kendall tau = {tau:.2f}, p = {pval:.3f}")
est.plot(marker="o", figsize=(10, 3), title="Chuva acumulada out–mar por safra (NASA POWER)", ylabel="mm");

## Recomendador de rotação

Compara **pelo menos três estratégias**, sempre **incluindo a rotação atual** do agricultor (as duas últimas culturas do histórico).
Cada estratégia é uma sequência **safra de grão → segundo cultivo** (safrinha, planta de cobertura ou pousio), plantando na data
informada e emendando o segundo cultivo logo após a colheita. Para cada uma, a ferramenta mostra:

- **Risco climático recente (2009–2024)** e quanto ele **mudou** desde 1991–2008;
- **Água que a lavoura usa** no ciclo (mm), **dias de calor extremo** e **risco de geada**, com os limites de temperatura do ZARC de cada cultura
  (ex.: feijão sofre acima de 32 °C; geada a partir de 2–3 °C de mínima);
- **Por quê** foi sugerida, os **benefícios** e o que **avaliar com o agrônomo** (pragas, mercado, manejo);
- **Onde faltam informações** no modelo.

A **nota final (0–100)** junta risco climático, saúde do solo e diversificação (o segundo cultivo traz algo novo?), com os pesos
das **prioridades** do agricultor. No fim, uma tabela mostra **quando plantar a soja** (normal ou precoce) para o milho safrinha
entrar dentro da janela segura.

**Parâmetros:** ciclos, limites de ISNA, profundidade de raiz, água no solo e temperaturas vêm das **portarias do ZARC**
(MAPA/Embrapa): soja MG 2025/26, milho 2ª safra DF, feijão 1ª safra MG e sorgo granífero. Crotalária e braquiária não têm ZARC
e usam estimativas.

In [ ]:
# ===== Funções do recomendador de rotação =====

# O que cada cultura traz e o que pede atenção (resumo agronômico geral; validar com agrônomo da região)
INFO_CULTURAS = {
    "soja": {"beneficio": "fixa nitrogênio do ar e tem mercado firme",
             "atencao": "repetir soja favorece ferrugem, nematoides e percevejos"},
    "soja_precoce": {"beneficio": "fixa nitrogênio e libera a área mais cedo para a safrinha",
                     "atencao": "ciclo curto rende um pouco menos; repetir soja favorece ferrugem e nematoides"},
    "milho_safrinha": {"beneficio": "deixa muita palha e raízes e gera renda na segunda safra",
                       "atencao": "plantio tardio aumenta o risco de seca no fim do ciclo; cigarrinha"},
    "sorgo": {"beneficio": "tolera mais a seca que o milho e tem raízes profundas",
              "atencao": "mercado menor: confirmar comprador na região"},
    "feijao": {"beneficio": "ciclo curto, fixa nitrogênio e gera renda",
               "atencao": "sensível a falta de água e a calor na floração; doenças de solo"},
    "crotalaria": {"beneficio": "fixa nitrogênio, ajuda a reduzir nematoides e cobre o solo",
                   "atencao": "não gera renda direta; custo da semente"},
    "braquiaria": {"beneficio": "raízes profundas e muita palha melhoram a infiltração; pode virar pasto",
                   "atencao": "precisa ser dessecada a tempo da próxima safra"},
}


LACUNAS = [
    "Ciclos, limites de ISNA, raiz e temperaturas vêm das portarias do ZARC; a divisão em fases e o Kc (FAO-56) são estimativas.",
    "Crotalária e braquiária não têm ZARC: seus parâmetros e as notas de saúde do solo são estimativas a validar com agrônomo.",
    "Não considera pragas, doenças, preços, máquinas nem disponibilidade de semente.",
    "O clima vem do NASA POWER, uma grade de ~50 km: pode diferir do que acontece na sua lavoura.",
    "Quando o solo é informado pelo tipo (arenoso, argiloso...), a classe de água é uma aproximação.",
]


def decendio_da_data(data):
    data = pd.Timestamp(data)
    return (data.month - 1) * 3 + min((data.day - 1) // 10, 2) + 1


def proximo_decendio(ano, k, dias):
    """Primeiro decêndio que começa depois de `dias` dias a partir do decêndio k do ano. Retorna (anos a somar, k)."""
    fim = datas_decendios(ano)[k - 1] + pd.Timedelta(days=dias)
    for a in (fim.year, fim.year + 1):
        for j, d in enumerate(datas_decendios(a), start=1):
            if d >= fim:
                return a - ano, j


def _linhas(sim, cultura, solo, k):
    return sim[(sim.cultura == cultura) & (sim.classe_solo == solo) & (sim.decendio == k)].set_index("ano")


def base(c):
    return CULTURAS[c]["base"] if c in CULTURAS else c


def extremos(df, etapas, anos):
    """etapas: [(cultura, decêndio, anos a somar)]. Média de dias acima do limite de calor de cada cultura
    e % de anos com mínima no limite de geada, dentro do ciclo de cada uma."""
    calor, geada = [], []
    for ano in anos:
        c_ano, g_ano, ok = 0, False, True
        for cult, k, da in etapas:
            n = sum(d for _, d, _ in CULTURAS[cult]["fases"])
            ini = datas_decendios(ano + da)[k - 1]
            jan = df.loc[ini: ini + pd.Timedelta(days=n - 1)]
            if len(jan) < n:
                ok = False
                break
            c_ano += int((jan.tmax >= CULTURAS[cult]["calor"]).sum())
            g_ano = g_ano or bool((jan.tmin <= CULTURAS[cult]["geada"]).any())
        if ok:
            calor.append(c_ano)
            geada.append(g_ano)
    return (float(np.mean(calor)) if calor else np.nan, float(np.mean(geada)) if geada else np.nan)


def risco_sequencia(sim, a, b, solo, k1):
    """Falha por ano (True/False) e água usada na sequência a → b plantando a no decêndio k1."""
    la = _linhas(sim, a, solo, k1)
    if la.empty:
        return None
    n_a = sum(d for _, d, _ in CULTURAS[a]["fases"])
    da, k2 = proximo_decendio(int(la.index.min()), k1, n_a)
    if b is None:
        return la.falha, la.agua_mm, k2, da
    lb = _linhas(sim, b, solo, k2)
    lb.index = lb.index - da          # alinha o 2º cultivo ao ano da safra principal
    j = la[["falha", "agua_mm"]].join(lb[["falha", "agua_mm"]], how="inner", rsuffix="_b")
    return j.falha | j.falha_b, j.agua_mm + j.agua_mm_b, k2, da   # falha se qualquer uma falhar


def recomendar_rotacao(sim, df, historico, solo, data_plantio, prioridades, top=3):
    """Compara estratégias safra de grão → segundo cultivo. Sempre inclui a rotação atual do agricultor.
    Diversificação = o 2º cultivo traz algo novo (a safra principal de renda pode continuar a mesma)."""
    k1 = decendio_da_data(data_plantio)
    hist = [base(c) for c in historico]
    ultima = hist[-1] if hist else None
    atual = tuple(hist[-2:]) if len(hist) >= 2 else None
    fam_ultima = CULTURAS[ultima]["familia"] if ultima in CULTURAS else None
    pesos = {k: prioridades.get(k, 0) for k in ("risco", "solo", "diversidade")}
    soma = sum(pesos.values()) or 1

    linhas = []
    k2 = None
    for a, ca in CULTURAS.items():
        e_atual_a = atual is not None and base(a) == atual[0]
        if ca["papel"] != "grão" or (base(a) == ultima and not e_atual_a):
            continue  # safra principal é de grão (renda) e não repete a última cultura
        for b in [None] + [c for c in CULTURAS if base(c) != base(a)]:
            res = risco_sequencia(sim, a, b, solo, k1)
            if res is None:
                continue
            f, agua, k2b, da = res
            r_rec = f[f.index.isin(PERIODO_2)].mean()
            r_ant = f[f.index.isin(PERIODO_1)].mean()
            if np.isnan(r_rec):
                continue
            k2 = k2 or k2b

            seq = [a] + ([b] if b else [])
            saude = np.mean([CULTURAS[c]["saude_solo"] for c in seq]) * (1 if b else 0.6)  # pousio deixa o solo descoberto
            if fam_ultima and CULTURAS[a]["familia"] != fam_ultima:
                saude += 0.10
            if b and CULTURAS[a]["familia"] != CULTURAS[b]["familia"]:
                saude += 0.10
            if any(CULTURAS[c]["papel"] == "cobertura" for c in seq):
                saude += 0.05
            saude = min(saude, 1.0)
            divers = 0.0 if b is None else float(base(b) not in hist)
            nota = 100 * (pesos["risco"] * (1 - r_rec) + pesos["solo"] * saude + pesos["diversidade"] * divers) / soma

            motivos = [f"risco de falta de água de {r_rec:.0%} nos últimos anos"]
            if fam_ultima and CULTURAS[a]["familia"] != fam_ultima:
                motivos.append(f"alterna com {CULTURAS[ultima]['nome'].lower()}")
            if any(CULTURAS[c]["familia"] == "leguminosa" for c in seq):
                motivos.append("tem leguminosa, que fixa nitrogênio")
            if any(CULTURAS[c]["papel"] == "cobertura" for c in seq):
                motivos.append("a cobertura protege o solo")
            if b is None:
                motivos.append("o solo fica descoberto depois da colheita")
            if any(CULTURAS[c]["fonte"] == "estimativa" for c in seq):
                motivos.append("parâmetros da cobertura são estimados")

            etapas = [(a, k1, 0)] + ([(b, k2b, da)] if b else [])
            calor, geada = extremos(df, etapas, PERIODO_2)
            linhas.append({
                "safra": ca["nome"],
                "depois": CULTURAS[b]["nome"] if b else "Pousio (solo descoberto)",
                "atual": atual is not None and (base(a), base(b) if b else None) == atual and a == historico[-2],
                "risco": r_rec,
                "mudanca_pp": (r_rec - r_ant) * 100,
                "agua_mm": agua[agua.index.isin(PERIODO_2)].mean(),
                "calor_dias": calor,
                "geada": geada,
                "saude_solo": saude,
                "diversidade": divers,
                "nota": nota,
                "motivo": "; ".join(motivos),
                "beneficios": "; ".join(INFO_CULTURAS[c]["beneficio"] for c in seq),
                "atencao": "; ".join(INFO_CULTURAS[c]["atencao"] for c in seq),
            })
    todas = pd.DataFrame(linhas).sort_values("nota", ascending=False).reset_index(drop=True)
    todas.index += 1                            # posição no ranking
    rec = todas[~todas.atual].head(top)
    rec = pd.concat([rec, todas[todas.atual]])  # a rotação atual entra sempre, como referência
    return rec, k1, k2


def janela_soja_milho(sim, solo, decendios=range(27, 34)):
    """Risco recente de soja → milho safrinha conforme a data de plantio da soja (set-3 a nov-3)."""
    linhas = []
    for k in decendios:
        lin = {"plantio da soja": rotulos[k - 1]}
        for s in ("soja_precoce", "soja"):
            f, _, k2, _ = risco_sequencia(sim, s, "milho_safrinha", solo, k)
            lin[f"{CULTURAS[s]['nome']}: milho entra em"] = rotulos[k2 - 1]
            lin[f"{CULTURAS[s]['nome']}: risco"] = f[f.index.isin(PERIODO_2)].mean()
        linhas.append(lin)
    return pd.DataFrame(linhas)

def semaforo(risco):
    """Cor pelo risco climático, nas faixas do ZARC: até 20% verde, até 40% amarelo, acima vermelho."""
    if risco <= 0.20:
        return "verde", "✓ Pode plantar", "#d9f2d9", "#1e6b2a"
    if risco <= 0.40:
        return "amarelo", "! Plantar com cuidado", "#fdf0c4", "#7a5a00"
    return "vermelho", "✕ Evite", "#f8d7d4", "#9b1c13"


def cartoes_html(rec):
    from html import escape
    cards = []
    for pos, r in rec.iterrows():
        _, rotulo, fundo, tinta = semaforo(r.risco)
        selo = ('<span style="background:#fff;border:1px solid #333;border-radius:10px;padding:1px 8px;'
                'font-size:12px;margin-left:6px">Sua rotação atual</span>') if r.atual else ""
        mud = f"{r.mudanca_pp:+.0f} p.p. desde 1991–2008"
        cards.append(f"""
<div style="background:{fundo};color:#1d1d1b;border-left:8px solid {tinta};border-radius:6px;padding:12px 14px;
            margin:8px 0;max-width:640px;font-family:system-ui,sans-serif;line-height:1.4">
  <div style="font-size:12px;color:{tinta};font-weight:700">{escape(rotulo)} · posição {pos} · nota {r.nota:.0f}/100</div>
  <div style="font-size:17px;font-weight:700;margin:2px 0 6px">{escape(r.safra)} → {escape(r.depois)}{selo}</div>
  <div style="font-size:13px">Falta de água: <b>{r.risco:.0%}</b> dos anos ({mud}) ·
       Água que a lavoura usa: <b>{r.agua_mm:.0f} mm</b> · Dias de calor extremo: <b>{r.calor_dias:.0f}</b> ·
       Risco de geada: <b>{r.geada:.0%}</b> dos anos</div>
  <div style="font-size:13px;margin-top:6px"><b>Por quê:</b> {escape(r.motivo)}.</div>
  <div style="font-size:13px"><b>Benefícios:</b> {escape(r.beneficios)}.</div>
  <div style="font-size:13px"><b>Avaliar com o agrônomo:</b> {escape(r.atencao)}.</div>
</div>""")
    lac = "".join(f"<li>{escape(x)}</li>" for x in LACUNAS)
    aviso = f"""
<div style="max-width:640px;font-family:system-ui,sans-serif;font-size:12px;color:inherit;margin-top:6px">
  <b>Onde faltam informações:</b><ul style="margin:4px 0 0 18px;padding:0">{lac}</ul>
  <p style="margin:6px 0 0">Esta ferramenta ajuda a enxergar padrões e opções. Ela <b>não substitui</b> a experiência do
  produtor nem a orientação de um agrônomo da região.</p>
</div>"""
    return "".join(cards) + aviso

In [ ]:
# ===== Painel do produtor =====
from IPython.display import HTML, display

rec, k1, k2 = recomendar_rotacao(sim, df, HISTORICO, SOLO_AD, DATA_PLANTIO, PRIORIDADES)
melhor = rec.iloc[0]

print(f"📍 {MUNICIPIO} | solo: {NOMES_AD[SOLO_AD]} | plantio: {rotulos[k1 - 1]} (decêndio {k1})")
print(f"🌱 Histórico: {' → '.join(CULTURAS[c]['nome'] for c in HISTORICO)}")
print(f"✅ Melhor opção: {melhor.safra} e depois {melhor.depois} (nota {melhor.nota:.0f}/100)\n")

# Cartões: cor = risco climático (verde ≤ 20%, amarelo ≤ 40%, vermelho > 40%); ordem = nota final
display(HTML(cartoes_html(rec)))

# Gráfico: nota de cada estratégia, separada nas três partes
w = {k: PRIORIDADES.get(k, 0) for k in ("risco", "solo", "diversidade")}
s = sum(w.values()) or 1
partes = pd.DataFrame({
    "Clima (menos risco)": 100 * w["risco"] * (1 - rec.risco.values) / s,
    "Saúde do solo": 100 * w["solo"] * rec.saude_solo.values / s,
    "Diversificação": 100 * w["diversidade"] * rec.diversidade.values / s,
}, index=(rec.safra + " → " + rec.depois + np.where(rec.atual, " (atual)", "")).values)
ax = partes.iloc[::-1].plot.barh(stacked=True, figsize=(10, 0.6 * len(partes) + 1.5),
                                 color=["#2a78b5", "#5a9a3a", "#d08c2c"])
ax.set_xlabel("Nota (0–100)")
ax.set_title(f"Estratégias comparadas — plantio em {rotulos[k1 - 1]}, {NOMES_AD[SOLO_AD]}")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

# Quando plantar a soja para o milho safrinha entrar a tempo? (janela do ZARC para o milho 2ª safra no Cerrado)
jan = janela_soja_milho(sim, SOLO_AD)
fmt = {c: "{:.0%}" for c in jan.columns if c.endswith("risco")}
print("\n🌽 Soja seguida de milho safrinha: risco conforme a data de plantio da soja")
display(jan.style.format(fmt).hide(axis="index"))

## Próximos passos
- **Calibrar** fases, Kc e limites de ISNA de cada cultura com a documentação do ZARC; revisar as notas de saúde do solo com agrônomos.
- **Validar:** comparar `tabua_risco_nasa_power.csv` com a tábua oficial do ZARC para o município (concordância das faixas por decêndio).
- **Solo local:** puxar a classe AD de mapas de solo (ex.: SoilGrids/Embrapa) a partir da coordenada, em vez de digitar.
- **Escalar:** rodar para vários municípios (lista de lat/lon) e montar um mapa.
- **Interface:** transformar o painel do produtor numa página simples (formulário → recomendação).